In [1]:
import mediapipe as mp
import cv2
import numpy as np
import os
import time
import tensorflow

In [2]:
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(min_detection_confidence=0.4,min_tracking_confidence=0.4)
drawing = mp.solutions.drawing_utils

In [3]:
def detection(image , model ):
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image,results


In [4]:
def draw(image,results):
    drawing.draw_landmarks(image,results.pose_landmarks,mp_holistic.POSE_CONNECTIONS,drawing.DrawingSpec((129,45,66),thickness=2,circle_radius=1),drawing.DrawingSpec((129,45,66),thickness=2,circle_radius=1))
    drawing.draw_landmarks(image,results.left_hand_landmarks,mp_holistic.HAND_CONNECTIONS)
    drawing.draw_landmarks(image,results.right_hand_landmarks,mp_holistic.HAND_CONNECTIONS)

In [5]:
def extract_landmarks(results):
    POSE_FEATURES = 33 * 2
    HAND_FEATURES = 21 * 2
    TOTAL_FEATURES = POSE_FEATURES + HAND_FEATURES + HAND_FEATURES
    if results.pose_landmarks:
        pose_landmark = results.pose_landmarks.landmark
        refer_x = pose_landmark[0].x
        refer_y = pose_landmark[0].y
    else:
        return np.zeros(TOTAL_FEATURES)
    
    def normalize(landmarks,refer_x,refer_y):
        features = []
        for lm in landmarks:
            norm_x = lm.x - refer_x
            norm_y = lm.y - refer_y
            features.extend([norm_x,norm_y])
        return np.array(features,dtype=np.float32)
    
    pose_featuers = normalize(pose_landmark,refer_x,refer_y)
    
    if results.left_hand_landmarks:
        left_hand_featuers = normalize(results.left_hand_landmarks.landmark,refer_x,refer_y)
    else:
        left_hand_featuers = np.zeros(HAND_FEATURES,dtype=np.float32)

    if results.right_hand_landmarks:
        right_hand_featuers = normalize(results.right_hand_landmarks.landmark,refer_x,refer_y)
    else:
        right_hand_featuers = np.zeros(HAND_FEATURES,dtype=np.float32)
    final = np.concatenate([pose_featuers,left_hand_featuers,right_hand_featuers]).astype(np.float32)
    return final

In [7]:
import cv2
import mediapipe as mp
import time

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def main(camera_index=0):
    # افتح الكاميرا مع flag خاص لويندوز لتقليل مشاكل
    cap = cv2.VideoCapture(camera_index, cv2.CAP_DSHOW)

    # استخدم نموذج Holistic داخل with ليتم إغلاقه تلقائياً
    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:
        prev_time = 0
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                print("لم يتم قراءة إطار من الكاميرا. إنهاء.")
                break

            # انعكاس/قلب الصورة لتجربة مرآة (اختياري)
            frame = cv2.flip(frame, 1)

            # تحويل BGR -> RGB ثم المعالجة
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image_rgb.flags.writeable = False
            results = holistic.process(image_rgb)
            image_rgb.flags.writeable = True

            # رسم العلامات على الإطار الأصلي (BGR)
            # استعمال mp_drawing.draw_landmarks للمظاهر المختلفة
            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    results.pose_landmarks,
                    mp_holistic.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2),
                    mp_drawing.DrawingSpec(color=(0,0,255), thickness=2)
                )

            if results.left_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    results.left_hand_landmarks,
                    mp_holistic.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(255,0,0), thickness=2, circle_radius=2),
                    mp_drawing.DrawingSpec(color=(255,0,0), thickness=2)
                )

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    results.right_hand_landmarks,
                    mp_holistic.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(0,128,255), thickness=2, circle_radius=2),
                    mp_drawing.DrawingSpec(color=(0,128,255), thickness=2)
                )

            # حساب واظهار FPS بسيط
            curr_time = time.time()
            fps = 1 / (curr_time - prev_time) if prev_time else 0
            prev_time = curr_time
            cv2.putText(frame, f"FPS: {int(fps)}", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

            cv2.imshow("MediaPipe Holistic - Press 'q' to quit", frame)

            # اضغط q للخروج (أو ESC)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q') or key == 27:
                break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

In [6]:
def collecter(guestur_name:str,no_videos:int):
    data_path = os.path.join("trining_data")
    actions = np.array([guestur_name])
    no_frames = 30

    for vid in range(no_videos):
        try:
            os.makedirs(os.path.join(data_path, guestur_name, str(vid)))
        except FileExistsError:
            pass

    cam = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    for video in range(no_videos):
        for frame in range(no_frames):
            success, image = cam.read()
            if not success:
                break
            image = cv2.flip(image, 1)
            image, results = detection(image, holistic)
            draw(image, results)

            if frame == 0: 
                cv2.putText(image, 'STARTING COLLECTION', (120,200), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
                cv2.putText(image, f'Collecting frames for {guestur_name} Video Number {video}', (15,12), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                cv2.imshow('OpenCV Feed', image)
                cv2.waitKey(3000)
            else: 
                cv2.putText(image, f'Collecting frames for {guestur_name} Video Number {video}', (15,12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                cv2.imshow('OpenCV Feed', image)


            features = extract_landmarks(results)
            path = os.path.join(data_path,str(guestur_name),str(video),str(frame))

            np.save(path,features)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
    cam.release()
    cv2.destroyAllWindows()

In [9]:
collecter("name3",25)

In [12]:
model = tensorflow.keras.models.load_model(r"C:\Users\Cloud\Downloads\MEDIAPIPE\.ipynb_checkpoints\test model 2 0.83.h5")

In [14]:
actions = np.array(["Good","THANKS","THINKING"]) 
frames = []

cam = cv2.VideoCapture(0, cv2.CAP_DSHOW)
display_until = 0
last_pred_time = 0
bond = 0.70
while cam.isOpened():
    success, frame = cam.read()
    if not success:
        break
    
    frame = cv2.flip(frame, 1)
    image, results = detection(frame, holistic)
    draw(image, results)
    landmarks = extract_landmarks(results)
    frames.append(landmarks)
    frames = frames[-30:]

    if len(frames) == 30 and (time.time() - last_pred_time) >= 2:
        prediction = model.predict(np.expand_dims(frames, axis=0), verbose=0) 
        confident = np.max(prediction)
        predicted_action = actions[np.argmax(prediction)]
        display_until = time.time() + 1
        last_pred_time = time.time()

    if time.time() <= display_until and predicted_action:
        if confident>=bond:
            cv2.putText(image, predicted_action, (200, 450), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 200), 2)
        else:
            prediction = " "
    cv2.imshow('holistic', image)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()